##### Copyright 2024 The IREE Authors

In [1]:
#@title Licensed under the Apache License v2.0 with LLVM Exceptions.
# See https://llvm.org/LICENSE.txt for license information.
# SPDX-License-Identifier: Apache-2.0 WITH LLVM-exception

# <img src="https://huggingface.co/datasets/huggingface/brand-assets/resolve/main/hf-logo.png" height="20px"> Hugging Face to <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/1/10/PyTorch_logo_icon.svg/640px-PyTorch_logo_icon.svg.png" height="20px"> PyTorch to <img src="https://raw.githubusercontent.com/iree-org/iree/main/docs/website/docs/assets/images/IREE_Logo_Icon_Color.svg" height="20px"> IREE

This notebook uses [iree-turbine](https://github.com/iree-org/iree-turbine) to export a pretrained [Hugging Face Transformers](https://huggingface.co/docs/transformers/) model to [IREE](https://github.com/iree-org/iree), leveraging [torch-mlir](https://github.com/llvm/torch-mlir) under the covers.

* The pretrained [whisper-small](https://huggingface.co/openai/whisper-small)
  model is showcased here as it is small enough to fit comfortably into a Colab
  notebook. Other pretrained models can be found at
  https://huggingface.co/docs/transformers/index.

## Setup

In [2]:
%%capture
#@title Uninstall existing packages
#   This avoids some warnings when installing specific PyTorch packages below.
!python -m pip uninstall -y fastai torchaudio torchdata torchtext torchvision

In [ ]:
# FIX: bumped torch 2.5.0 -> 2.7.1
!python -m pip install --pre --index-url https://download.pytorch.org/whl/cpu --upgrade torch==2.7.1

Looking in indexes: https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 174.6/174.6 MB 105.9 MB/s  0:00:0100:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 72.4 MB/s  0:00:00
  Attempting uninstall: sympy
    Found existing installation: sympy 1.14.0
    Uninstalling sympy-1.14.0:
      Successfully uninstalled sympy-1.14.0
  Attempting uninstall: torch━━━━━━━━━━━━━━━━━━━ 0/2 [sympy]
    Found existing installation: torch 2.7.1+cpu 0/2 [sympy]
    Uninstalling torch-2.7.1+cpu:╺━━━━━━━━━━━━━━━━━━━ 1/2 [torch]
      Successfully uninstalled torch-2.7.1+cpu━━━━━━━━━━━━━━━━ 1/2 [torch]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [torch]32m1/2 [torch]


In [4]:
!python -m pip install --pre iree-turbine -f https://iree.dev/pip-release-links.html

Looking in links: https://iree.dev/pip-release-links.html


In [5]:
#@title Report version information
!echo "Installed iree-turbine, $(python -m pip show iree_turbine | grep Version)"

!echo -e "\nInstalled IREE, compiler version information:"
!iree-compile --version

import torch
print("\nInstalled PyTorch, version:", torch.__version__)

Installed iree-turbine, Version: 3.10.0rc20260601

Installed IREE, compiler version information:
IREE (https://iree.dev):
  IREE compiler version 3.12.0rc20260601 @ d7c7b98e5de3ae574b87516b81ff489e0f49e7ee
  LLVM version 23.0.0git
  Optimized build

Installed PyTorch, version: 2.5.0+cpu


## Load and run whisper-small

Load the pretrained model from https://huggingface.co/openai/whisper-small.

See also:

* Model card: https://huggingface.co/docs/transformers/model_doc/whisper
* Test case in [AMD-SHARK-TestSuite](https://github.com/nod-ai/AMD-SHARK-TestSuite/): [`pytorch/models/whisper-small/model.py`](https://github.com/nod-ai/AMD-SHARK-TestSuite/blob/main/e2eamdshark/pytorch/models/whisper-small/model.py)

In [7]:
# FIX: added cell -- install latest Transformers (required by the updated model-loading code below)
!python -m pip install --pre transformers --upgrade

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# https://huggingface.co/docs/transformers/model_doc/auto
# AutoModelForCausalLM -> WhisperForCausalLM
# AutoTokenizer        -> WhisperTokenizerFast

# FIX: workaround for Transformers v5 + torch-mlir/IREE export.
# Transformers v5 builds attention masks via `masking_utils`, whose packed-sequence
# detection calls `torch.diff` (see `find_packed_sequence_indices`). torch-mlir cannot
# legalize `torch.aten.diff`, so compilation fails. For ordinary (non-packed)
# single-sequence input this detection is a no-op (returns None during normal
# execution), so we disable it to keep the exported graph free of `torch.diff`.
# We also keep `attn_implementation="eager"`: torch-mlir cannot legalize the fused
# `scaled_dot_product_attention` op that the "sdpa" path emits.
import transformers.masking_utils as masking_utils
masking_utils.find_packed_sequence_indices = lambda position_ids: None  # FIX

modelname = "openai/whisper-small"
tokenizer = AutoTokenizer.from_pretrained(modelname)

# Some of the options here affect how the model is exported. See the test cases
# at https://github.com/nod-ai/AMD-SHARK-TestSuite/tree/main/e2eamdshark/pytorch/models
# for other options that may be useful to set.
model = AutoModelForCausalLM.from_pretrained(
    modelname,
    output_attentions=False,
    output_hidden_states=False,
    attn_implementation="eager",
    # FIX: `torchscript=True` was removed in Transformers v5; its export-relevant
    # effect was disabling the KV cache, so use `use_cache=False` instead. Otherwise
    # the model returns a DynamicCache, which is not pytree-flattenable and
    # torch.export() fails.
    use_cache=False,
)

# This is just a simple demo to get some data flowing through the model.
# Depending on this model and what input it expects (text, image, audio, etc.)
# this might instead use a specific Processor class. For Whisper,
# WhisperProcessor runs audio input pre-processing and output post-processing.
example_prompt = "Hello world!"
example_encoding = tokenizer(example_prompt, return_tensors="pt")
example_input = example_encoding["input_ids"].cpu()
example_args = (example_input,)

Test exporting using [`torch.export()`](https://pytorch.org/docs/stable/export.html#torch.export.export). If `torch.export` works, `aot.export()` from Turbine should work as well.

In [ ]:
import torch
exported_program = torch.export.export(model, example_args)

Export using the simple [`aot.export()`](https://iree.dev/guides/ml-frameworks/pytorch/#simple-api) API from Turbine.

In [ ]:
import iree.turbine.aot as aot
# Note: aot.export() wants the example args to be unpacked.
whisper_compiled_module = aot.export(model, *example_args)

Compile using Turbine/IREE then run the program.

In [ ]:
binary = whisper_compiled_module.compile(save_to=None)

import iree.runtime as ireert
config = ireert.Config("local-task")
vm_module = ireert.load_vm_module(
    ireert.VmModule.wrap_buffer(config.vm_instance, binary.map_memory()),
    config,
)

iree_outputs = vm_module.main(example_args[0])
# FIX: recent iree.runtime returns NumPy arrays directly; older versions returned
# device arrays needing an explicit `.to_host()`. Handle both.
iree_output = iree_outputs[0]
print(iree_output.to_host() if hasattr(iree_output, "to_host") else iree_output)

Run the program using native PyTorch to compare outputs.

In [ ]:
torch_outputs = model(example_args[0])
print(torch_outputs[0].detach().numpy())

Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


[[[  5.8126183    3.9667587    4.5749483  ...   2.7658575    2.643694
     1.5479784 ]
  [  7.563436     6.029952     5.100036   ...   6.4327083    6.101557
     6.4348083 ]
  [  0.93802685  -4.469646    -4.012787   ...  -6.2486415   -7.7918167
    -6.8453975 ]
  [  0.74507916  -3.763197    -7.487034   ...  -6.734877    -6.966276
   -10.022424  ]
  [ -0.96288276  -3.510221    -6.0158725  ...  -7.1164136   -6.708687
   -10.225745  ]
  [  3.3470666    2.492654    -3.304323   ...  -1.5709934   -1.8455791
    -2.9992423 ]]]
